In [ ]:
!pip install -q groq faiss-cpu sentence-transformers pandas tqdm transformers

In [ ]:
import sys
import os

# Add project root to path so modules are importable
sys.path.insert(0, os.path.abspath(".."))

from config import client, embed_model, BASE_MODEL, JUDGE_MODEL
from data.loader import load_tatqa
from indexing.vector_store import build_vector_store
from pipeline import run_comparison
from evaluation.evaluator import evaluate_all, summarize_results

In [ ]:
# Load data — update path to your local TAT-QA file
DATA_PATH = "../tatqa_dataset_train.json"
samples = load_tatqa(DATA_PATH, max_samples=100)
print(f"Loaded {len(samples)} samples")

In [ ]:
# Build baseline index (text only)
baseline_index, baseline_chunks, baseline_metadata = build_vector_store(
    samples, embed_model, use_table_aware_chunking=False
)

# Build improved index (text + table-aware chunking)
improved_index, improved_chunks, improved_metadata = build_vector_store(
    samples, embed_model, use_table_aware_chunking=True
)

In [ ]:
# Run a single comparison on the first sample question
sample_question = samples[0]["question"]

t1, t2, t3 = run_comparison(
    question=sample_question,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="improved",
    top_k=10,
)

In [ ]:
# Evaluate baseline
baseline_df = evaluate_all(
    samples=samples,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="baseline",
    max_samples=15,
    top_k=6,
)

# Evaluate improved
improved_df = evaluate_all(
    samples=samples,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="improved",
    max_samples=15,
    top_k=6,
)

In [ ]:
import pandas as pd

# Build and display comparison summary
baseline_summary = summarize_results(baseline_df, label="Baseline (text only)")
improved_summary = summarize_results(improved_df, label="Improved (table-aware)")

comparison_df = pd.concat([baseline_summary, improved_summary], ignore_index=True)
comparison_df